# Pneumonia Classification from Chest X-rays

**Computer Vision Assignment 2

Goal: Improve a baseline CNN to maximise accuracy, precision, and recall when classifying chest X-ray images as NORMAL or PNEUMONIA.

Dataset: Chest X-ray images train/test split provided

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Check GPU availability
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import time

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D, Rescaling
from tensorflow.keras.optimizers import Adam

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Dataset paths - UPDATE THESE to match your Google Drive structure
TRAIN_DIR = '/content/drive/MyDrive/CV_Assignment/chest_xray/train'
TEST_DIR  = '/content/drive/MyDrive/CV_Assignment/chest_xray/test'

# Hyperparameters (from lecturer's baseline script)
BATCH_SIZE = 12
EPOCHS = 8
IMG_HEIGHT = 128
IMG_WIDTH  = 128
IMG_CHANNELS = 3

In [ ]:
# Load training data with 80/20 train/val split
train_ds, val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    seed=123,
    validation_split=0.2,
    subset='both',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    labels='inferred',
    shuffle=True
)

# Load test data
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_DIR,
    seed=None,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    labels='inferred',
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)
print('Class names:', class_names)
print('Number of classes:', num_classes)

In [ ]:
# Display a few sample images from the training set
plt.figure(figsize=(12, 6))
for images, labels in train_ds.take(1):
    for i in range(6):
        ax = plt.subplot(2, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i].numpy()])
        plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Build the baseline CNN
model = Sequential([
    Rescaling(1.0/255, input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)),
    Conv2D(16, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=Adam(),
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Train and measure training time
start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

training_time = time.time() - start_time
print(f'\nTraining completed in {training_time:.1f} seconds ({training_time/60:.1f} minutes)')

In [ ]:
# Plot accuracy and loss curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

# Loss
axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(test_ds)
print(f'Test accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

In [ ]:
# Show predictions on a few test images
test_batch = test_ds.take(1)
plt.figure(figsize=(12, 6))
for images, labels in test_batch:
    for i in range(6):
        ax = plt.subplot(2, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))

        # Make prediction
        prediction = model.predict(tf.expand_dims(images[i], 0), verbose=0)
        pred_class = class_names[np.argmax(prediction)]
        true_class = class_names[labels[i].numpy()]
        confidence = 100 * np.max(prediction)

        plt.title(f'True: {true_class}\nPred: {pred_class} ({confidence:.1f}%)')
        plt.axis('off')
plt.tight_layout()
plt.show()